<a href="https://colab.research.google.com/github/YudiH11/MSSP6070/blob/master/WeeklyModules/Week10/Week_10_Improved_Image_Downloader.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 10 Participation Activity: Improving Python Code

**Original notebook:** Week 9 — Download images from Openverse.

**Goal:** Search Openverse for cat photos from Flickr and download up to five JPEG images into a local `images` folder.

### AI suggestions used
1. Use the `keyword` and `max_downloads` settings throughout the code instead of repeating fixed values.
2. Remove unused imports and repeated `res.json()` calls.
3. Show clear results with `print()` rather than expressions that may not display all messages.
4. Handle download errors and check that saved `.jpg` files are actually JPEG images.
5. Stop after the requested number of successful downloads and list the saved files.

The code still uses the same Openverse API and the same basic steps as the Week 9 notebook.

In [ ]:
import os
import time
import requests

## 1. Set the search options
Change `keyword` or `max_downloads` to search for different images or download a different number.

In [ ]:
keyword = "cats"
max_downloads = 5
folder = "images"

os.makedirs(folder, exist_ok=True)

## 2. Search Openverse
Ask for more results than we need because some image links may not work or may not be JPEG files.

In [ ]:
url = "https://api.openverse.org/v1/images/"
params = {
    "q": keyword,
    "page_size": min(max_downloads * 3, 20),
    "source": "flickr"
}

try:
    response = requests.get(url, params=params, timeout=30)
    response.raise_for_status()
    images = response.json().get("results", [])
    print("Images found:", len(images))
except (requests.RequestException, ValueError) as error:
    print("Search failed:", error)
    images = []

Images found: 15


## 3. Download JPEG images
Only JPEG responses are saved with a `.jpg` extension. A failed link does not stop the remaining downloads.

In [ ]:
downloaded = 0

for image in images:
    if downloaded >= max_downloads:
        break

    image_url = image.get("url")
    if not image_url:
        continue

    try:
        print("Downloading:", image_url)
        image_response = requests.get(image_url, timeout=30, stream=True)
        image_response.raise_for_status()

        # A .jpg filename should contain a JPEG image.
        content_type = image_response.headers.get("Content-Type", "").lower()
        if "image/jpeg" not in content_type:
            print("Skipped: not a JPEG image")
            image_response.close()
            continue

        filename = f"cat_{downloaded + 1}.jpg"
        filepath = os.path.join(folder, filename)

        with open(filepath, "wb") as image_file:
            for chunk in image_response.iter_content(chunk_size=100000):
                if chunk:
                    image_file.write(chunk)

        downloaded += 1
        print("Saved:", filepath)

    except requests.RequestException as error:
        print("Download failed:", error)
    finally:
        if "image_response" in locals():
            image_response.close()
            del image_response

    time.sleep(2)

print(f"Downloaded {downloaded} of {max_downloads} requested images.")

Downloading: https://live.staticflickr.com/3313/3481540500_c846c62863_b.jpg
Saved: images/cat_1.jpg
Downloading: https://live.staticflickr.com/2706/5840168826_486f364c6c_b.jpg
Saved: images/cat_2.jpg
Downloading: https://live.staticflickr.com/3436/3717404325_db41d8d687_b.jpg
Saved: images/cat_3.jpg
Downloading: https://live.staticflickr.com/7300/11748899763_2dd16da8b4_b.jpg
Saved: images/cat_4.jpg
Downloading: https://live.staticflickr.com/7004/6494613787_9a3031fa7d_b.jpg
Saved: images/cat_5.jpg
Downloaded 5 of 5 requested images.


## 4. Check downloaded files
Only files beginning with `cat_` and ending with `.jpg` are included in the final list.

In [ ]:
files = sorted(
    name for name in os.listdir(folder)
    if name.startswith("cat_") and name.lower().endswith(".jpg")
)

print("JPEG files in folder:", len(files))
print(files)

JPEG files in folder: 5
['cat_1.jpg', 'cat_2.jpg', 'cat_3.jpg', 'cat_4.jpg', 'cat_5.jpg']


### Submission notes
- Download this improved `.ipynb` file and upload it to **PARTICIPATION ACTIVITY: Week 10 Improving Coding** in Canvas.
- Run all cells in Google Colab or Jupyter with internet access to generate the `images` folder.
- There is **no input CSV or other dataset** for this notebook. The code gets search results from the Openverse API and downloads image files. If your instructor wants the downloaded images as well, download the `images` folder (or zip it) from your notebook environment.
- Results depend on the live API and whether the image URLs are available.